# GLOF Detection — GLOFeagles '26 Challenge
## Glacial Lake Outburst Flood Detection using YOLOv8 + CBAM Attention

This notebook demonstrates the full pipeline:
1. Dataset preparation
2. Model architecture (YOLOv8m + CBAM)
3. Training
4. Inference & Evaluation
5. Segmentation mask generation

## 1. Setup & Installation

In [ ]:
!pip install ultralytics torch torchvision opencv-python numpy matplotlib pyyaml

## 2. CBAM Attention Module

The **Convolutional Block Attention Module (CBAM)** consists of:
- **Channel Attention:** Learns inter-channel relationships via avg/max pooling + shared MLP
- **Spatial Attention:** Learns inter-spatial relationships via channel-wise avg/max pooling + conv

Reference: Woo et al., "CBAM: Convolutional Block Attention Module", ECCV 2018

In [ ]:
import torch
import torch.nn as nn

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        reduced_channels = max(1, channels // reduction)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, reduced_channels, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(reduced_channels, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = self.fc(self.avg_pool(x))
        mx = self.fc(self.max_pool(x))
        return self.sigmoid(avg + mx)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg, mx], dim=1)
        return self.sigmoid(self.conv(x))


class CBAM(nn.Module):
    """Convolutional Block Attention Module"""
    def __init__(self, c1, c2=None):
        super().__init__()
        self.ca = ChannelAttention(c1)
        self.sa = SpatialAttention()

    def forward(self, x):
        x = self.ca(x) * x
        x = self.sa(x) * x
        return x

# Quick test
test_input = torch.randn(1, 512, 20, 20)
cbam = CBAM(512)
test_output = cbam(test_input)
print(f'CBAM Input shape:  {test_input.shape}')
print(f'CBAM Output shape: {test_output.shape}')
print(f'Shapes match: {test_input.shape == test_output.shape}')

## 3. Dataset Overview

| Split | Images | Bounding Boxes |
|-------|--------|---------------|
| Train | 210 | 1,850 |
| Valid | 30 | 309 |
| Test | 16 | — |

### Classes (7)
| ID | Class | Count |
|----|-------|-------|
| 0 | cloud | 20 |
| 1 | debris | 565 |
| 2 | debris and snow | 367 |
| 3 | lake | 215 |
| 4 | snow | 297 |
| 5 | terrain shadow | 358 |
| 6 | waterflow | 28 |

In [ ]:
import os
from collections import Counter

CLASSES = ['cloud', 'debris', 'debris and snow', 'lake', 'snow', 'terrain shadow', 'waterflow']
labels_dir = 'training/dataset/train/labels'

class_counts = Counter()
total_boxes = 0

for f in os.listdir(labels_dir):
    if f.endswith('.txt'):
        with open(os.path.join(labels_dir, f)) as fh:
            for line in fh:
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_counts[int(parts[0])] += 1
                    total_boxes += 1

print(f'Total bounding boxes: {total_boxes}')
for cls_id in sorted(class_counts.keys()):
    print(f'  Class {cls_id} ({CLASSES[cls_id]}): {class_counts[cls_id]} boxes')

## 4. Model Training

We inject the CBAM module into the YOLOv8m architecture by monkey-patching the Ultralytics parser.

In [ ]:
from ultralytics import YOLO
import ultralytics.nn.modules
import ultralytics.nn.tasks

# Inject CBAM into Ultralytics
ultralytics.nn.modules.CBAM = CBAM
ultralytics.nn.tasks.CBAM = CBAM

# Build model from custom YAML
model = YOLO('yolov8-cbam.yaml')

# Load pretrained COCO weights for transfer learning
model.load('yolo11m.pt')

# Train
results = model.train(
    data='training/dataset/glof.yaml',
    epochs=30,
    batch=16,
    imgsz=640,
    device=0,
    project='runs',
    name='glof_yolov8m_cbam'
)

print('Training complete!')

## 5. Inference

In [ ]:
from ultralytics import YOLO
import ultralytics.nn.modules
import ultralytics.nn.tasks
import cv2
import matplotlib.pyplot as plt

# Inject CBAM
ultralytics.nn.modules.CBAM = CBAM
ultralytics.nn.tasks.CBAM = CBAM

# Load trained model
model = YOLO('best.pt')

# Run inference on a test image
results = model('training/dataset/test/images/', conf=0.45)

# Display first result
if results:
    res_plotted = results[0].plot()
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB))
    plt.title('GLOF Detection Result')
    plt.axis('off')
    plt.show()

## 6. Segmentation Mask Generation

In [ ]:
import numpy as np
import os

def generate_segmentation_mask(image, results, num_classes=7):
    h, w = image.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    if results and len(results) > 0:
        boxes = results[0].boxes
        if boxes is not None:
            for box in boxes:
                cls_id = int(box.cls[0]) + 1
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                mask[max(0,y1):min(h,y2), max(0,x1):min(w,x2)] = cls_id
    return mask

def colorize_mask(mask):
    palette = {
        0: (0,0,0), 1: (255,255,255), 2: (0,128,128),
        3: (0,200,200), 4: (255,200,0), 5: (200,200,220),
        6: (80,80,80), 7: (255,100,50)
    }
    h, w = mask.shape
    color = np.zeros((h, w, 3), dtype=np.uint8)
    for cid, c in palette.items():
        color[mask == cid] = c
    return color

# Generate masks for all test images
os.makedirs('segmentation_masks', exist_ok=True)
test_dir = 'training/dataset/test/images'

for fname in os.listdir(test_dir):
    if fname.lower().endswith(('.jpg','.png','.jpeg')):
        img = cv2.imread(os.path.join(test_dir, fname))
        res = model(img, conf=0.45)
        mask = generate_segmentation_mask(img, res)
        cv2.imwrite(f'segmentation_masks/{os.path.splitext(fname)[0]}_mask.png', mask)
        color = colorize_mask(mask)
        cv2.imwrite(f'segmentation_masks/{os.path.splitext(fname)[0]}_color.png', color)

print(f'Masks generated for {len(os.listdir("segmentation_masks"))//2} images')

## 7. Evaluation Metrics Summary

| Metric | Description |
|--------|------------|
| **IoU** | Intersection over Union (primary metric) |
| **Precision** | TP / (TP + FP) |
| **Recall** | TP / (TP + FN) |
| **F1 Score** | 2 × (P × R) / (P + R) |
| **Cohen's Kappa** | Agreement beyond chance |

### Deployment
- **Frontend:** React.js on Netlify — [glof26.netlify.app](https://glof26.netlify.app)
- **Backend:** FastAPI on Hugging Face Spaces